In [9]:
from pathlib import Path
import numpy as np
import nibabel as nib
import pyvista as pv
import torch
from nibabel.nifti1 import Nifti1Image
from generate_4d_heart import CavityLabel

partial_labels = (CavityLabel.LV_MYO, CavityLabel.LV, CavityLabel.RV)

root_dir = Path.cwd().parent

physic_phantom_dir = Path("/media/E/sj/Data/physic_phantom")
cavity_partial_path = physic_phantom_dir / "label.nii.gz"
image_path = physic_phantom_dir/ "phantom_500FOV.nii.gz"
coronary_path = physic_phantom_dir / "coronary.nii.gz"

ssm_dir = root_dir / "generate_4d_heart" /"ssm" / "ssm_world"
ssm_template_path = ssm_dir / "ssm_template.vtk"
b_motion_path = ssm_dir / "b_motion.npy"
P_motion_path = ssm_dir / "P_motion.npy"

(temp_output := Path("temp/Phantom")).mkdir(parents=True, exist_ok=True)


def read_nii(path: Path) -> tuple[Nifti1Image, np.ndarray]:
    img = nib.load(path)
    assert isinstance(img, Nifti1Image)
    return img, img.get_fdata()

cavity_img, cavity_data = read_nii(cavity_partial_path)
image_img, image_data = read_nii(image_path)
coronary_img, coronary_data = read_nii(coronary_path)
cavity_data = cavity_data.round().astype(np.int8)
coronary_data = coronary_data.round().astype(np.int8)

ssm_template: pv.PolyData = pv.read(ssm_template_path)  #type: ignore

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
image_tensor = torch.from_numpy(image_data).to(device, torch.float32)[None, None]  # (1, 1, *shape)
cavity_tensor = torch.from_numpy(cavity_data).to(device, torch.int64)[None, None]  # (1, 1, *shape)

assert isinstance(ssm_template, pv.PolyData)
b_motion: np.ndarray = np.load(b_motion_path)
P_motion: np.ndarray = np.load(P_motion_path)


In [10]:
new_mesh_list = []
for label in partial_labels:
    submesh = ssm_template.extract_values(label, scalars="label").extract_surface(algorithm=None)
    new_mesh_list.append(submesh)

new_mesh = pv.merge(new_mesh_list)
ssm_template = new_mesh.copy()


In [11]:
import cupy as cp
import pyacvd
from cupyx.scipy import ndimage
from generate_4d_heart import NUM_TOTAL_POINTS, CavityLabel
from generate_4d_heart.ssm.ssm import pytorch3d_refine
from generate_4d_heart.ssm.polydata_io import apply_affine
import torchcpd
import torch


def _get_largest_connected_component(data: np.ndarray) -> cp.ndarray:
    """
    Obtain the largest connected region in the binary image. Copy from Phantom/script/get_surface_cloud.py
    Args:
        data: the binary image
    Returns:
        np.ndarray: the largest connected region in the binary image
    """
    data_cp = cp.asarray(data)
    labeled_data, num_features = ndimage.label(data_cp)  # type: ignore
    if num_features == 1:
        return data_cp
    sizes = ndimage.sum(data_cp, labeled_data, cp.arange(num_features + 1))  # type: ignore
    largest_component = sizes.argmax()
    return (labeled_data == largest_component).astype(cp.uint8)


def get_surface_from_label(
        label: np.ndarray,
        affine: np.ndarray
) -> pv.PolyData:
    """
    Get the cloud from the label. Copy from Phantom/script/get_surface_cloud.py
    Args:
        nii_path: the path to the nii file
        max_point_num: the maximum number of points in the cloud for each label
    Returns:
        pv.PolyData: the cloud
        np.ndarray: the affine matrix of the nii file
    """
    surface_all = pv.PolyData()
    
    label_cp = cp.asarray(label)
    for label_id in partial_labels:
        assert label_id > 0
        if label_id == CavityLabel.LV_MYO:
            # The myocardial part of the left ventricle is combined with the left ventricular cavity part
            mask = cp.zeros(label_cp.shape, dtype=cp.uint8)
            mask[label_cp == int(CavityLabel.LV_MYO)] = 1
            mask[label_cp == int(CavityLabel.LV)] = 1
            mask = (mask > 0).astype(cp.uint8)
        else:
            mask = (label_cp == int(label_id)).astype(cp.uint8)
        
        structure = ndimage.generate_binary_structure(3, 3)
        mask = ndimage.binary_closing(mask, iterations=1, structure=structure)
        mask = ndimage.binary_opening(mask, iterations=1, structure=structure)
        mask = _get_largest_connected_component(mask)
        mask: np.ndarray = cp.asnumpy(mask)

        surface = pv.wrap(mask).contour([1], method="flying_edges").triangulate().smooth_taubin().triangulate().clean()
        cluster = pyacvd.Clustering(surface)
        cluster.cluster(NUM_TOTAL_POINTS)
        surface = cluster.create_mesh().triangulate().clean()
        assert isinstance(surface, pv.PolyData)
        if np.isnan(surface.points).any():
            raise ValueError(f"NaN in points")
        if not surface.is_manifold:
            raise ValueError(f"Mesh is not manifold")
        surface.point_data["label"] = np.ones(surface.n_points).astype(np.uint8) * label_id
        surface.cell_data["label"] = np.ones(surface.n_cells).astype(np.uint8) * label_id
        surface_all = surface_all.merge(surface)
    
    assert isinstance(surface_all, pv.PolyData)
    surface_all.points = apply_affine(surface_all.points, affine)
    return surface_all

def deform_surface(
        source_surface: pv.PolyData, 
        target_surface: pv.PolyData, 
        device: int = 0,
        **deform_kwargs
    ) -> tuple[pv.PolyData, np.ndarray]:
    """
    Use deformable registration to deform the source surface to the target surface. Copy from Phantom/script/align_surface.py
    Args:
        source_surface: the source surface
        target_surface: the target surface
        device: the device to use
        deform_kwargs: the kwargs for deformable registration
    Returns:
        pv.PolyData: the deformed surface
        np.ndarray: the transformation matrix from the source surface to the target surface
    """
    source_surface = source_surface.copy()
    target_surface = target_surface.copy()
    new_cloud = pv.PolyData()
    rigid_reg = torchcpd.RigidRegistration(X=target_surface.points, Y=source_surface.points, device=device)
    new_points_all, _ = rigid_reg.register()
    s, R, t1 = rigid_reg.get_registration_parameters()
    affine_reg = torchcpd.AffineRegistration(X=target_surface.points, Y=new_points_all.cpu().numpy(), device=device)
    new_points_all, _ = affine_reg.register()
    B, t2 = affine_reg.get_registration_parameters()
    
    # TODO 将template配准到surface得到landmark时，所得的变换应该同步作用到template各点的deform vecotr上
    A1 = np.eye(4)
    A1[:3, :3] = (s * R.T).cpu().numpy() #type: ignore
    A1[:3, 3] = t1.cpu().numpy()  #type: ignore
    A2 = np.eye(4)
    A2[:3, :3] = B.T.cpu().numpy()  #type: ignore
    A2[:3, 3] = t2.cpu().numpy()  #type: ignore
    A_template2landmark = A2 @ A1
    
    
    # new_points_all, _ = torchcpd.RigidRegistration(X=target_surface.points, Y=source_surface.points, device=device).register()
    # new_points_all, _ = torchcpd.AffineRegistration(X=target_surface.points, Y=new_points_all.cpu().numpy(), device=device).register()
    
    source_surface.points = new_points_all.cpu().numpy()
    for label in sorted(partial_labels):
        source_submesh: pv.PolyData = source_surface.extract_values(label, scalars="label", preference="cell").extract_surface(algorithm=None)    # type: ignore
        target_submesh: pv.PolyData = target_surface.extract_values(label, scalars="label", preference="cell").extract_surface(algorithm=None)    # type: ignore
        new_points, _ = torchcpd.AffineRegistration(X=target_submesh.points, Y=source_submesh.points, device=device).register()
        new_points, _ = torchcpd.DeformableRegistration(X=target_submesh.points, Y=new_points.cpu().numpy(), device=device, kwargs=deform_kwargs).register()
        source_submesh.points = new_points.cpu().numpy()
        
        source_submesh = pytorch3d_refine(
            source_pv=source_submesh, 
            target_pv=target_submesh, 
            device=f"cuda:{device}",
            steps=200,
        )
        
        new_cloud = new_cloud.merge(source_submesh)
    
    return new_cloud, A_template2landmark

In [12]:
assert cavity_img.affine is not None
label_surface = get_surface_from_label(cavity_data, cavity_img.affine)
landmark_surface, A_template2landmark = deform_surface(ssm_template, label_surface.copy(), 0)
label_surface.save(temp_output / "label_surface.vtk")

In [13]:
plotter = pv.Plotter(shape=(1, 3), notebook=False)
plotter.camera_position = pv.CameraPosition(
    (-68.16251727594782, 104.066792868929, 344.7257297397248),
    (0, 0, 0),
    (0.7523987560658962, -0.5745229979148703, 0.3222102368600382)
)
c = np.array(label_surface.center)
plotter.subplot(0, 0)
plotter.add_mesh(ssm_template, scalars="label")
plotter.subplot(0, 1)
plotter.add_mesh(label_surface.translate(-c), scalars="label")
plotter.subplot(0, 2)
plotter.add_mesh(landmark_surface.translate(-c), scalars="label")
plotter.link_views()
plotter.show()

In [14]:
import einops
from generate_4d_heart.ssm import Landmark

def mesh_size(mesh: pv.PolyData) -> np.ndarray:
    bounds = np.array(mesh.bounds).reshape(3, 2)
    size = np.abs(bounds[:, 1] - bounds[:, 0])
    return size

num_components_used: int = 1
motion_multiplier: float = 1.0
n_phases = 20
b = b_motion
P = P_motion

c = num_components_used
k = motion_multiplier
deforms: np.ndarray = k * np.einsum('LPC, LCND -> LPND', b[:, :, :c], P[:, :c, :, :])
deforms = deforms @ A_template2landmark[:3, :3].T
# Lable is 1-based, but the index in b and P is 0-based, so need to minus 1.
partial_label_i = [int(label_id)-1 for label_id in partial_labels]
deforms = deforms[partial_label_i]
deforms = einops.rearrange(deforms, 'l p n d -> p (l n) d')

deformed_cavities: list[pv.PolyData] = []
for phase in range(n_phases):
    m = landmark_surface.copy()
    # Lable is 1-based, but the index in b and P is 0-based, so need to minus 1.
    m.points += deforms[phase]
    deformed_cavities.append(m)

landmark = Landmark(landmark_surface, deforms=deforms)

In [15]:
plotter = pv.Plotter(notebook=False, off_screen=False)
plotter.open_gif(str(temp_output / "moving.gif"), fps=30)
for cavity in deformed_cavities:
    plotter.clear()
    plotter.add_mesh(cavity, scalars="label", opacity=0.5)
    plotter.write_frame()
plotter.show()

In [57]:
from generate_4d_heart.rotate_dsa.data_reader.rbf_reader import KDTreeRBF
from generate_4d_heart.roi import ROI
import torch
from monai.networks.blocks.warp import Warp

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
rbf = KDTreeRBF(landmark).to(device).eval()
affine = cavity_img.affine
assert affine is not None
roi = ROI.get_from_cavity_np(cavity_data, affine)

shape_zoomed = roi.shape_after_crop_and_zoom
affine_zoomed = roi.affine_after_crop_and_zoom
grid_zoomed = np.meshgrid(
    np.arange(shape_zoomed[0]), 
    np.arange(shape_zoomed[1]), 
    np.arange(shape_zoomed[2]), 
    indexing="ij"
)
coords_zoomed = np.stack(grid_zoomed, axis=-1)  # (N, 3)
coords_zoomed = apply_affine(coords_zoomed, affine_zoomed)
coords_zoomed = torch.from_numpy(coords_zoomed).to(device, torch.float32)

affine_crop = torch.from_numpy(roi.affine_after_crop).to(device, torch.float32)
affine_crop_inv: torch.Tensor = torch.linalg.inv(affine_crop)

warp_label = Warp(mode="nearest", padding_mode="border").to(device).eval()
warp_image = Warp(mode="bilinear", padding_mode="border").to(device).eval()

/media/E/sj/Code/generate_4d_heart/.pixi/envs/default/lib/python3.13/site-packages/monai/networks/blocks/warp.py:72: UserWarning: monai.networks.blocks.Warp: Using PyTorch native grid_sample.
  warnings.warn("monai.networks.blocks.Warp: Using PyTorch native grid_sample.")


In [58]:
padding_voxel = 1
cavity_all_mask = np.isin(cavity_data, partial_labels).astype(np.uint8)
cavity_all_mask = cp.asnumpy(_get_largest_connected_component(cavity_all_mask))
cavity_add_coronary_mask = np.logical_or(cavity_all_mask, coronary_data > 0).astype(np.uint8)

# 2. 提取等值面 (Marching Cubes)
cavity_all_mesh: pv.PolyData = pv.wrap(cavity_add_coronary_mask)\
    .contour([1], method="flying_edges")\
    .triangulate()\
    .smooth_taubin()\
    .decimate_pro(
        reduction=0.5,          # 减少 三角面片
        preserve_topology=True, # 防止破洞
    )\
    .clean()

points = cavity_all_mesh.points
normals = cavity_all_mesh.point_normals
offset_points = points + normals * padding_voxel

pericardium_mesh = pv.PolyData(offset_points)\
    .delaunay_3d()\
    .extract_surface(algorithm=None)

ref_volume = pv.ImageData(dimensions=cavity_all_mask.shape)
pericardium_mask = pericardium_mesh\
    .voxelize_binary_mask(reference_volume = ref_volume)\
    .point_data['mask']\
    .reshape(cavity_all_mask.shape, order='F')

pericardium_mask = pericardium_mask.astype(np.uint8)

In [59]:
pl = pv.Plotter(notebook=False)
for label in partial_labels:
    mask = (cavity_data == label).astype(np.uint8)
    mesh = pv.wrap(mask).contour([1], method="flying_edges").triangulate().smooth_taubin(n_iter=50).decimate_pro(
        reduction=0.95,          # 减少 三角面片
        preserve_topology=True, # 防止破洞
        feature_angle=30.0
    ).clean()
    pl.add_mesh(mesh, name=f"Label {label}", color=pv.Color(np.random.rand(3)), opacity=0.8)

coronary_mesh = pv.wrap(coronary_data).contour([1], method="flying_edges").triangulate().smooth_taubin(n_iter=50)
pl.add_mesh(coronary_mesh, name="Coronary", color="red", opacity=0.8)
pl.add_mesh(pericardium_mesh.extract_all_edges(), name="Pericardium", color="red")  #type: ignore
pl.show()

In [60]:
attenuation_transition = 10
pericardium_mask_cropped = roi.crop_on_data(pericardium_mask)
with cp.cuda.Device(device.index):
    external_dist = ndimage.distance_transform_edt(1 - cp.asarray(pericardium_mask_cropped))
    attenuation_mask = cp.asnumpy(cp.exp(-external_dist / attenuation_transition))  # 从心包外部开始，距离越远运动衰减越强，距离为transition时衰减为1/e #type: ignore
attenuation_mask = torch.from_numpy(attenuation_mask).to(device, torch.float32)[None, None]  # (1, 1, *shape_after_crop)


In [61]:
from einops import rearrange, einsum
from typing import Literal
from torch.nn import functional as F
from generate_4d_heart.rotate_dsa.data_reader.rbf_reader import invert_displacement_field
from generate_4d_heart.rotate_dsa.cardiac_phase import CardiacPhase

def warp(
    phase: CardiacPhase, 
    image_i: torch.Tensor, 
    cavity_i: torch.Tensor, 
    coronary_i: torch.Tensor,
    do_recover: bool = False
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    dx: torch.Tensor = rbf(phase, coords_zoomed)  # (*shape_zoomed, 3) in world coordinate
    dx = rearrange(dx, "x y z c -> 1 c x y z")
    dx = F.interpolate(
        dx, 
        size=roi.shape_after_crop, 
        mode="trilinear", 
        align_corners=False
    )  # (1, 3, *shape_after_crop)  in world coordinate

    R_world_to_voxel = affine_crop_inv[:3, :3]        
    dx = einsum(dx, R_world_to_voxel, "b c x y z, c i -> b i x y z")  # in voxel coordinate

    dx = dx * attenuation_mask  # Attenuate the movement away from the pericardium

    dx = invert_displacement_field(dx)  # forward deformation field -> inverse deformation field, still in voxel coordinate
    
    def warp_recover(tensor: torch.Tensor, mode: Literal["image", "label"]) -> torch.Tensor:
        if mode == "image":
            warp = warp_image
            recover = lambda x: roi.recover_cropped_tensor(x, background=image_tensor)
            final_dtype = torch.float32
        elif mode == "label":
            warp = warp_label
            recover = roi.recover_cropped_tensor
            final_dtype = torch.uint8
        else:
            raise ValueError(f"Unsupported mode: {mode}")
        
        out = warp(tensor.to(device, torch.float32), dx)
        if do_recover:
            out = recover(out)
        return out.to(final_dtype)
            
    return warp_recover(image_i, mode="image"), warp_recover(cavity_i, mode="label"), warp_recover(coronary_i, mode="label")
        

In [62]:
from matplotlib import pyplot as plt
from generate_4d_heart.saver import save_nii, save_gif
from cupyx.scipy.ndimage import distance_transform_edt

image_output_dir = temp_output / "warped_images"
cavity_output_dir = temp_output / "warped_cavities"
image_output_dir.mkdir(parents=True, exist_ok=True)
cavity_output_dir.mkdir(parents=True, exist_ok=True)
assert affine is not None


image_cropped = roi.crop_on_data(image_tensor)
cavity_cropped = roi.crop_on_data(cavity_tensor)
coronary_cropped = roi.crop_on_data(torch.from_numpy(coronary_data).to(device, torch.int64)[None, None])

slices = []
for phase_i in range(n_phases):
    image_warped, cavity_warped, coronary_warped = warp(
        CardiacPhase.from_index(phase_i, n_phases), 
        image_cropped, 
        cavity_cropped, 
        coronary_cropped, 
        do_recover=False
    )
    print(f"Phase {phase_i}: image warped shape: {image_warped.shape}, cavity warped shape: {cavity_warped.shape}")
    save_nii(image_output_dir/f"{phase_i:02d}_image_warped.nii.gz", image_warped, affine)
    save_nii(cavity_output_dir/f"{phase_i:02d}_cavity_warped.nii.gz", cavity_warped, affine, is_label=True)
    save_nii(cavity_output_dir/f"{phase_i:02d}_coronary_warped.nii.gz", coronary_warped, affine, is_label=True)
    slices.append(image_warped.squeeze().cpu().numpy()[..., image_warped.shape[-1]//2])

save_gif(temp_output / "warped_images.gif", frames=np.stack(slices), cmap="gray")

Phase 0: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 1: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 2: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 3: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 4: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 5: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 6: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 7: image warped shape: torch.Size([1, 1, 125, 125, 123]), cavity warped shape: torch.Size([1, 1, 125, 125, 123])
Phase 8: image warped shape: torch.Size([1, 1, 1

In [63]:
# from matplotlib import pyplot as plt
# from generate_4d_heart.saver import save_nii, save_gif
# from cupyx.scipy.ndimage import distance_transform_edt

# image_output_dir = temp_output / "warped_images"
# cavity_output_dir = temp_output / "warped_cavities"
# image_output_dir.mkdir(parents=True, exist_ok=True)
# cavity_output_dir.mkdir(parents=True, exist_ok=True)
# assert affine is not None

# # mask: 1 表示保留区域，0 表示需要填充
# mask: np.ndarray = pericardium_mask > 0
# mask_cropped = roi.crop_on_data(mask) > 0
# masked_image = image_data[~mask] = -1000

# pericardium_mask_tensor = torch.from_numpy(pericardium_mask).to(device, torch.bool)[None, None]
# pericardium_mask_cropped_tensor = torch.from_numpy(pericardium_mask_cropped).to(device, torch.bool)[None, None]
# masked_image_tensor = torch.from_numpy(masked_image).to(device, torch.float32)[None, None]

# image_cropped = roi.crop_on_data(image_tensor)
# image_masked_cropped = roi.crop_on_data(masked_image_tensor)
# cavity_cropped = roi.crop_on_data(cavity_tensor)
# coronary_cropped = roi.crop_on_data(torch.from_numpy(coronary_data).to(device, torch.int64)[None, None])

# slices = []
# for phase_i in range(10):
#     image_warped, cavity_warped, coronary_warped = warp(
#         CardiacPhase.from_index(phase_i, n_phases), 
#         image_masked_cropped, 
#         cavity_cropped, 
#         coronary_cropped, 
#         do_recover=False
#     )
#     image_warped[~pericardium_mask_cropped_tensor] = image_cropped[~pericardium_mask_cropped_tensor]
    
#     print(f"Phase {phase_i}: image warped shape: {image_warped.shape}, cavity warped shape: {cavity_warped.shape}")
#     save_nii(image_output_dir/f"{phase_i:02d}_image_warped.nii.gz", image_warped, affine)
#     save_nii(cavity_output_dir/f"{phase_i:02d}_cavity_warped.nii.gz", cavity_warped, affine, is_label=True)
#     save_nii(cavity_output_dir/f"{phase_i:02d}_coronary_warped.nii.gz", coronary_warped, affine, is_label=True)
#     slices.append(image_warped.squeeze().cpu().numpy()[..., image_warped.shape[-1]//2])

# save_gif(temp_output / "warped_images.gif", frames=np.stack(slices), cmap="gray")

In [64]:
import json
corp_box_json = {
    "crop_box": roi.to_dict()["crop_box"],
}
with open(temp_output / "crop_box.json", "w") as f:
    json.dump(corp_box_json, f)